# 🕸️ DeepFetch
**Run Cell 1 once per session, then Cell 2 to start the server.**  
Your public dashboard link appears at the end of Cell 2.


In [ ]:
# Cell 1 — System setup (once per session, ~5 min)
import subprocess, os

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise RuntimeError(f'FAILED: {cmd}')
    return r.stdout.strip()

# 1. Node.js 22
print('📦 Node.js 22...')
sh('curl -fsSL https://deb.nodesource.com/setup_22.x | bash - 2>&1 | tail -2')
sh('apt-get install -y nodejs 2>&1 | tail -2')
print(f'   ✅ Node {sh("node --version")}  npm {sh("npm --version")}')

# 2. Build tools for better-sqlite3 (native C++ addon)
print('📦 Build tools...')
sh('apt-get install -y --no-install-recommends build-essential python3 python3-dev make g++ 2>&1 | tail -2')
print('   ✅ done')

# 3. Playwright system libs
print('📦 Playwright libs...')
sh('apt-get install -y --no-install-recommends '
   'libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 '
   'libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 '
   'libpango-1.0-0 libpangocairo-1.0-0 2>&1 | tail -2')
print('   ✅ done')

# 4. cloudflared
print('📦 cloudflared...')
sh('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 '
   '-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
print(f'   ✅ {sh("cloudflared --version").split()[2]}')

# 5. Wipe npm config + cache — Colab may inherit stale Replit registry settings
print('🧹 Cleaning npm config and cache...')
sh('rm -f /root/.npmrc /root/.npm/_npmrc 2>/dev/null; echo ok')
sh('npm config set registry https://registry.npmjs.org')
sh('npm config delete proxy      2>/dev/null; echo ok')
sh('npm config delete https-proxy 2>/dev/null; echo ok')
sh('npm cache clean --force 2>&1 | tail -1')
print(f'   ✅ registry={sh("npm config get registry")}')

print()
print('🎉 Ready — run Cell 2.')


In [ ]:
# Cell 2 — Clone · Build · Launch  (re-run anytime to restart)
import subprocess, os, time, re, secrets

def sh(cmd, cwd=None, show=0):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-4000:])
        raise RuntimeError(f'FAILED ({r.returncode}): {cmd}')
    if show:
        lines = (r.stdout + r.stderr).splitlines()
        if lines: print('\n'.join(lines[-show:]))
    return r.stdout.strip()

REPO = '/deepfetch'
SRV  = f'{REPO}/server'
DASH = f'{REPO}/dashboard'
DIST = f'{SRV}/dist/index.js'
TSC  = f'{SRV}/node_modules/.bin/tsc'
VITE = f'{DASH}/node_modules/.bin/vite'
PW   = '/ms-playwright'
PORT = 3000
DATA = '/deepfetch-data'
CFG  = f'{REPO}/config.yaml'
NPM  = 'npm install --registry https://registry.npmjs.org --no-audit --no-fund'

os.makedirs(DATA, exist_ok=True)
os.environ['PLAYWRIGHT_BROWSERS_PATH'] = PW

# Always enforce public registry (in case Cell 1 wasn't run this session)
sh('npm config set registry https://registry.npmjs.org')

# Kill old processes
subprocess.run("pkill -f 'node.*dist/index'; pkill -f cloudflared",
               shell=True, capture_output=True)
time.sleep(1)

# 1. Clone / pull
if not os.path.exists(REPO):
    print('📥 Cloning...')
    sh(f'git clone --depth 1 https://github.com/ferelking242/deepfetch {REPO}')
else:
    print('📥 Pulling latest...')
    sh('git pull --ff-only', cwd=REPO)

# 2. npm install — wipe node_modules if tsc not present (stale/partial install)
if not os.path.exists(TSC):
    # Clean any partial state that may cache bad registry URLs
    if os.path.exists(f'{SRV}/node_modules'):
        print('🧹 Removing stale server node_modules...')
        sh(f'rm -rf {SRV}/node_modules')
    print('📦 npm install (server) — compiling native addon ~2 min...')
    sh(f'{NPM} 2>&1', cwd=SRV, show=5)
else:
    print('📦 server node_modules ✅')

if not os.path.exists(f'{DASH}/node_modules'):
    print('📦 npm install (dashboard)...')
    sh(f'{NPM} 2>&1', cwd=DASH, show=3)
else:
    print('📦 dashboard node_modules ✅')

# 3. Playwright Chromium
has_pw = os.path.exists(PW) and any(d.startswith('chromium') for d in os.listdir(PW))
if not has_pw:
    print('🎭 Installing Playwright Chromium...')
    sh(f'node {SRV}/node_modules/.bin/playwright install chromium --with-deps 2>&1', show=4)
    print('   ✅ ready')
else:
    print('🎭 Playwright Chromium ✅')

# 4. Build TypeScript server
if not os.path.exists(DIST):
    print('🔨 Building server...')
    sh(f'node {TSC} --project tsconfig.json 2>&1', cwd=SRV, show=10)
    if not os.path.exists(DIST):
        raise RuntimeError(f'{DIST} missing after build')
    print('   ✅ built')
else:
    print('🔨 Server dist ✅')

# 5. Build dashboard
if not os.path.exists(f'{DASH}/dist/index.html'):
    print('🔨 Building dashboard...')
    sh(f'node {VITE} build 2>&1', cwd=DASH, show=4)
    print('   ✅ built')
else:
    print('🔨 Dashboard ✅')

# 6. Config
if not os.path.exists(CFG):
    try: import yaml
    except: sh('pip install pyyaml -q'); import yaml
    with open(CFG, 'w') as f:
        yaml.dump({
            'server':   {'port': PORT, 'host': '0.0.0.0', 'master_secret': secrets.token_hex(32)},
            'browser':  {'pool_max': 0, 'pool_reserved': 1, 'context_ttl_seconds': 300,
                         'navigation_timeout_ms': 30000, 'headless': True},
            'resources':{'cpu_threshold_pct': 85, 'ram_threshold_pct': 80},
            'queue':    {'max_retries': 3, 'retry_base_delay_ms': 2000, 'result_ttl_seconds': 86400},
            'ai_engine':{'enabled': True, 'trigger': 'on_selector_failure',
                         'max_html_chars': 50000, 'timeout_ms': 15000,
                         'providers': [
                             {'name':'ollama','local':True,'model':'llama3.2','base_url':'http://localhost:11434'},
                             {'name':'groq',  'api_key':'','model':'llama-3.3-70b-versatile'},
                             {'name':'gemini','api_key':'','model':'gemini-2.0-flash'},
                             {'name':'openai','api_key':'','model':'gpt-4o-mini'},
                         ]},
            'zeusdl':   {'binary': 'zeusdl', 'extra_flags': []},
            'sessions': {'encryption_key': secrets.token_hex(32), 'check_interval_seconds': 1800},
            'data_dir': DATA,
        }, f, default_flow_style=False)
    print('⚙️  config.yaml written ✅')
else:
    print('⚙️  config.yaml ✅')

# 7. Start server
print('🚀 Starting server...')
srv_log = open('/tmp/deepfetch.log', 'w')
subprocess.Popen(['node', DIST],
    env={**os.environ, 'DF_CONFIG': CFG, 'NODE_ENV': 'production'},
    stdout=srv_log, stderr=srv_log, cwd=REPO)

import urllib.request
for i in range(40):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/v1/health', timeout=2)
        print(f'   ✅ up in {i+1}s')
        break
    except: pass
else:
    print('❌ Server logs:'); print(open('/tmp/deepfetch.log').read()[-4000:])
    raise RuntimeError('Server did not start')

# 8. Tunnel
print('🌐 Opening tunnel...')
cf = open('/tmp/cf.log', 'w')
subprocess.Popen(['cloudflared','tunnel','--url',f'http://localhost:{PORT}'],
                 stdout=cf, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(30):
    time.sleep(1)
    try:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', open('/tmp/cf.log').read())
        if m: PUBLIC_URL = m.group(0); break
    except: pass

if not PUBLIC_URL:
    PUBLIC_URL = f'http://localhost:{PORT}'
    print('⚠️  tunnel URL not found')

print(f'''
═══════════════════════════════════════════════════
  🎉  DeepFetch is LIVE
═══════════════════════════════════════════════════
  📊  Dashboard : {PUBLIC_URL}/dashboard
  📖  API Docs  : {PUBLIC_URL}/docs
  ❤️   Health    : {PUBLIC_URL}/v1/health
═══════════════════════════════════════════════════
  Open on any device — phone, tablet, PC
''')
